### Cross-vehicle consistency checks for NTD cases

The *baseline* scenario for neural tube defect (NTD) cases should not depend on vehicle for each country. This notebook verifies two properties of the NTD model outputs (`ntd_cases_by_scenario.csv`) for each country:

**Check 1: Zero scenario equals baseline when baseline effective coverage is zero**
- For each country/vehicle pair, we load the baseline effective folate coverage from `0100_data_prep`.
- If baseline coverage is 0 (no vehicle fortification at baseline), then the `zero` and `baseline` NTD cases must be identical. We assert this and print a confirmation.

**Check 2: Baseline NTD cases agree across vehicles within a country**
- The first vehicle's results for each country are used as a reference. Every other vehicle's `baseline` NTD cases must match that reference.
- The check is skipped for India because baseline rice fortification coverage is added on top of the GBD estimates, so India's baseline differs between vehicles. This should be revisited so baseline handling is consistent across vehicles.

In [13]:
import glob
import pandas as pd
from lsff_utils import paths
from pathlib import Path
import numpy as np

In [14]:
ntd_path = Path("../../../0500_neural_tube_defects_model/results/")

arms = sorted(
    (p.parts[-2], p.parts[-1])
    for p in ntd_path.glob("*/*")
    if (p / "ntd_cases_by_scenario.csv").exists()
)
arms = pd.DataFrame(arms, columns = ["country", "vehicle"])

In [15]:
for country in arms.country.unique():
    vehicle_ntd_paths = glob.glob(f"../../../0500_neural_tube_defects_model/results/{country}/*/ntd_cases_by_scenario.csv")
    first_vehicle_values = pd.read_csv(vehicle_ntd_paths[0])
    first_vehicle_values = first_vehicle_values[first_vehicle_values.scenario.isin(["zero", "baseline"])]
    first_vehicle_values = first_vehicle_values.set_index([d for d in first_vehicle_values.columns if d != "value"]).value

    for path in vehicle_ntd_paths:
        vehicle = Path(path).parts[-2]
        baseline = pd.read_csv(f"../../../0100_data_prep/results/folate/{vehicle}/baseline_fortification/effective_coverage/{country}.csv")
        baseline = baseline.set_index([c for c in baseline.columns if c != "value"]).value
        
        vehicle_values = pd.read_csv(path)
        vehicle_values = vehicle_values[vehicle_values.scenario.isin(["zero", "baseline"])]
        vehicle_values = vehicle_values.set_index([d for d in vehicle_values.columns if d != "value"]).value

        # no partial check exists if only some quintiles are 0
        if (baseline == 0).all():
            assert (vehicle_values.xs("zero", level = "scenario") == vehicle_values.xs("baseline", level = "scenario")).all()
            print(f"Zero and baseline matched for {country}/{vehicle}.")
        else:
            print(f"Baseline fortification exists for {country}/{vehicle}.")

        # this check doesn't pass in india due to baseline rice coverage adding to GBD estimates; this should be revisted for consistency in baseline across vehicles
        if country != "india":
            assert np.allclose(
                vehicle_values.xs("baseline", level = "scenario"), 
                first_vehicle_values.xs("baseline", level = "scenario"),
                rtol = 0,
                atol = 0.000001
            )
    if country != "india":        
        print(f"Baseline cases matched for all vehicles in {country}.")
        




Zero and baseline matched for ethiopia/wheat.
Zero and baseline matched for ethiopia/salt.
Baseline cases matched for all vehicles in ethiopia.
Baseline fortification exists for india/rice.
Zero and baseline matched for india/wheat.
Zero and baseline matched for nigeria/rice.
Baseline fortification exists for nigeria/wheat.
Zero and baseline matched for nigeria/salt.
Zero and baseline matched for nigeria/bouillon.
Baseline cases matched for all vehicles in nigeria.
